# Custom Losses for Autonomous Driving Simulation

This companion notebook matches the article `Custom Losses: A 1-Hour Interview Learning Session`.

You will run three small experiments:

1. Weighted cross entropy on imbalanced classification.
2. Focal loss on hard vs easy examples.
3. Why MSE averages multi-modal futures, and how multi-modal loss helps.

Runtime: CPU is fine, GPU is supported.

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

## Part 1: Weighted cross entropy

Toy task: classify normal driving vs rare cut-in.

The rare class is only about 5% of the data. We compare unweighted CE and weighted CE.

In [ ]:
def make_imbalanced_data(n=4000, rare_frac=0.05):
    n1 = int(n * rare_frac)
    n0 = n - n1
    # normal cluster
    x0 = torch.randn(n0, 2) * 1.0 + torch.tensor([0.0, 0.0])
    y0 = torch.zeros(n0, dtype=torch.long)
    # rare cut-in cluster, overlapping enough to make classification non-trivial
    x1 = torch.randn(n1, 2) * 1.0 + torch.tensor([1.5, 1.5])
    y1 = torch.ones(n1, dtype=torch.long)
    x = torch.cat([x0, x1], dim=0)
    y = torch.cat([y0, y1], dim=0)
    perm = torch.randperm(n)
    return x[perm].to(device), y[perm].to(device)

x, y = make_imbalanced_data()
print('class counts:', torch.bincount(y.cpu()))

plt.figure(figsize=(5, 4))
plt.scatter(x[y == 0, 0].cpu(), x[y == 0, 1].cpu(), s=4, alpha=0.2, label='normal')
plt.scatter(x[y == 1, 0].cpu(), x[y == 1, 1].cpu(), s=8, alpha=0.8, label='cut-in')
plt.legend(); plt.title('Imbalanced toy data'); plt.show()

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x):
        return self.net(x)

def train_classifier(weight=None, steps=400, lr=1e-2):
    model = TinyClassifier().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        opt.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y, weight=weight)
        loss.backward()
        opt.step()
    return model

def metrics(model):
    with torch.no_grad():
        pred = model(x).argmax(dim=-1)
    out = {}
    for cls in [0, 1]:
        tp = ((pred == cls) & (y == cls)).sum().item()
        fp = ((pred == cls) & (y != cls)).sum().item()
        fn = ((pred != cls) & (y == cls)).sum().item()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        out[cls] = (precision, recall)
    return out

unweighted = train_classifier(weight=None)
class_weights = torch.tensor([1.0, 10.0], device=device)
weighted = train_classifier(weight=class_weights)

print('unweighted metrics {class: (precision, recall)}:', metrics(unweighted))
print('weighted metrics   {class: (precision, recall)}:', metrics(weighted))

Discussion prompt:

- Did rare-class recall improve?
- Did rare-class precision drop?
- Why is that tradeoff expected?

## Part 2: Focal loss

Focal loss downweights easy examples using `(1 - p_t)^gamma`.

In [ ]:
def focal_loss(logits, targets, alpha=None, gamma=2.0):
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()
    row = torch.arange(targets.numel(), device=targets.device)
    pt = probs[row, targets]
    log_pt = log_probs[row, targets]
    alpha_t = 1.0 if alpha is None else alpha[targets]
    return (-alpha_t * (1 - pt).pow(gamma) * log_pt).mean()

toy_logits = torch.tensor([[4.0, -1.0], [0.2, 0.0], [-1.0, 3.0]], device=device)
toy_targets = torch.tensor([0, 0, 1], device=device)
probs = F.softmax(toy_logits, dim=-1)
pt = probs[torch.arange(3, device=device), toy_targets]
print('p_t:', pt.detach().cpu().numpy())
print('modulating factor gamma=2:', ((1 - pt) ** 2).detach().cpu().numpy())
print('CE:', F.cross_entropy(toy_logits, toy_targets).item())
print('focal:', focal_loss(toy_logits, toy_targets, gamma=2.0).item())

Takeaway: if the model is already confident and correct, focal loss makes that example contribute less gradient.

## Part 3: Why MSE fails for multi-modal futures

We create a toy intersection. From the same current state, the future is either straight or left. A one-mode MSE model learns the average.

In [ ]:
def make_trajectory_batch(n=1024, T=20):
    # Same input for all examples: current scene feature says 'at intersection'.
    cond = torch.ones(n, 1)
    t = torch.linspace(0, 1, T)
    straight = torch.stack([5 * t, torch.zeros_like(t)], dim=-1)
    left = torch.stack([2.5 * t, 4 * t], dim=-1)
    modes = torch.randint(0, 2, (n,))
    y = torch.where(modes[:, None, None] == 0, straight[None], left[None])
    y = y + 0.05 * torch.randn_like(y)
    return cond.to(device), y.to(device), modes.to(device)

cond, traj, modes = make_trajectory_batch()
T = traj.size(1)

plt.figure(figsize=(5, 4))
for i in range(80):
    color = 'tab:blue' if modes[i].item() == 0 else 'tab:orange'
    plt.plot(traj[i, :, 0].cpu(), traj[i, :, 1].cpu(), color=color, alpha=0.15)
plt.title('Two valid future modes'); plt.axis('equal'); plt.show()

In [ ]:
class OneModeTraj(nn.Module):
    def __init__(self, T):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, T * 2))
        self.T = T
    def forward(self, c):
        return self.net(c).view(-1, self.T, 2)

model = OneModeTraj(T).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(1000):
    opt.zero_grad()
    pred = model(cond)
    loss = F.mse_loss(pred, traj)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred = model(cond[:1])[0].cpu()

plt.figure(figsize=(5, 4))
for i in range(80):
    plt.plot(traj[i, :, 0].cpu(), traj[i, :, 1].cpu(), color='gray', alpha=0.08)
plt.plot(pred[:, 0], pred[:, 1], 'r-', linewidth=3, label='MSE one-mode prediction')
plt.legend(); plt.axis('equal'); plt.title('MSE learns the average future'); plt.show()

The red trajectory is not either real mode. In a driving scene, this can become an invalid path.

## Part 4: Multi-modal trajectory loss

Now predict K trajectories and mode probabilities. We train the closest mode to match the logged future.

In [ ]:
class MultiModeTraj(nn.Module):
    def __init__(self, T, K=2):
        super().__init__()
        self.K, self.T = K, T
        self.traj = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, K * T * 2))
        self.logits = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, K))
    def forward(self, c):
        B = c.size(0)
        return self.traj(c).view(B, self.K, self.T, 2), self.logits(c)

def multimodal_trajectory_loss(pred_trajs, mode_logits, target, cls_weight=0.2):
    diff = pred_trajs - target[:, None, :, :]
    sq_error = (diff ** 2).sum(dim=(-1, -2))
    best_mode = sq_error.argmin(dim=1)
    row = torch.arange(target.size(0), device=target.device)
    reg = sq_error[row, best_mode].mean()
    cls = F.cross_entropy(mode_logits, best_mode)
    return reg + cls_weight * cls

mm = MultiModeTraj(T, K=2).to(device)
opt = torch.optim.Adam(mm.parameters(), lr=1e-2)
for step in range(1500):
    opt.zero_grad()
    pred_trajs, mode_logits = mm(cond)
    loss = multimodal_trajectory_loss(pred_trajs, mode_logits, traj)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_trajs, mode_logits = mm(cond[:1])
    pred_trajs = pred_trajs[0].cpu()
    probs = F.softmax(mode_logits[0], dim=-1).cpu()

plt.figure(figsize=(5, 4))
for i in range(80):
    plt.plot(traj[i, :, 0].cpu(), traj[i, :, 1].cpu(), color='gray', alpha=0.08)
for k in range(2):
    plt.plot(pred_trajs[k, :, 0], pred_trajs[k, :, 1], linewidth=3, label=f'mode {k}, p={probs[k]:.2f}')
plt.legend(); plt.axis('equal'); plt.title('Multi-modal model learns both futures'); plt.show()

## Interview drills

1. Explain why weighted CE scales gradients for rare classes.
2. Explain why focal loss is about hard examples, not just rare classes.
3. Explain why MSE learns the conditional mean.
4. Implement `multimodal_trajectory_loss` from memory.
5. Name two failure modes of multi-modal trajectory losses.